# 공유속성 M5: 기존 체크포인트의 N/V 작동 진단

Dunnhumby seed48·300epoch·개발 DAY684~690 체크포인트 **하나만 읽습니다. 새 학습 0회**이며 마지막주 최종 test·H&M·holdout은 열지 않습니다. GPU 런타임에서 위에서부터 모두 실행하세요.

비교는 현재 전체 표현 / 같은 공동학습 모델의 ID 표현만 / 공유속성을 유지하고 M2의 실제 N/V 튜플만 고객 사이에서 섞은 표현입니다. ID-only는 M1이 아닙니다. 원본 전체142지표를 먼저 재현하고, 정답 진입·이탈과 전체·CLV세그먼트·상품지지도·가격대 결과를 저장합니다.

N/V는 최근365일 반복거래율과 거래당 평균금액의 백분위입니다. 학습 degree 10분위 안에서 유효 고객의 두 값을 공동순열하며, 무효 고객·M3·M4는 유지합니다. 순열 seed1048을 한 번 사용하고 결과가 좋을 때까지 다시 섞지 않습니다.

이 비교는 고정 모델의 추론 반응입니다. N/V가 학습된 ID에 남긴 간접효과는 제거하지 않습니다. 단일 반복노출 개발시드·단일 순열이며 유의성·인과효과·다음 학습의 성공을 주장하지 않습니다. 결과를 보기 전에는 강도·차원·epoch를 바꾸지 않습니다.

## 1. Drive 연결과 고정 코드
기존 완료 결과와 체크포인트를 읽기만 합니다. 진단 결과는 Colab 로컬의 새 폴더에 저장하고 마지막 셀에서 ZIP을 다운로드합니다. Drive에 결과를 쓰지 않습니다. 런타임 종료 전에 ZIP을 내려받으세요.

In [ ]:
from google.colab import drive
from pathlib import Path
import json, os, subprocess, sys, tempfile

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
SOURCE_COMMIT = '0779d6cfded45f1afda1ee0390f328c5ac50e559'
REPO = Path('/content/clv-m5-shared-checkpoint-' + SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', SOURCE_COMMIT], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', SOURCE_COMMIT], check=True)
assert subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip() == SOURCE_COMMIT
os.chdir(REPO)
sys.path.insert(0, str(REPO))
import torch
assert torch.cuda.is_available(), 'GPU 런타임으로 변경하세요. 학습이 아니라 읽기·추론 진단입니다.'
import clv_m5_shared_attributes_checkpoint_diagnostic as diagnostic
SOURCE_DIR = Path(diagnostic.DEFAULT_SOURCE)
OUT_DIR = Path(tempfile.mkdtemp(prefix='clv-m5-shared-attributes-checkpoint-seed48-', dir='/content'))
print('GPU:', torch.cuda.get_device_name(0))
print('원본:', SOURCE_DIR)
print('로컬 진단 출력:', OUT_DIR)
print('신규 학습 0회; 원본 결과·체크포인트 읽기만 수행')

## 2. 전체 지표 재현 → 세 가지 상태 비교
원본의 입력·설정·출처·300epoch 및 M3/M4 가중치와 체크포인트 입력 버퍼를 검사합니다. 체크포인트가 없으면 재학습하지 않고 중단합니다.

현재 전체 표현의142지표가 재현돼야 나머지 비교를 진행합니다. 일반 지표 허용오차는 rtol=1e-5·atol=1e-7이며, 기존 코드와 동일하게 가격대 정렬도만 atol=1e-5입니다. 실패하면 readback.csv를 확인하고 임의로 허용오차를 넓히지 마세요.

고정 ID·아이템 표현은 한 번 계산해 그대로 공유합니다. N/V 비교는 실제 전체이력 표현에 N/V로 바뀐 메시지 차이만 결정적으로 집계해 반영합니다. CUDA의 반복 전파·원래 이력 합산 차이를 N/V 효과로 오인하지 않도록 하며, 원래 full 표현과 전체 지표 재현 검사는 유지합니다.

In [ ]:
try:
    paths = diagnostic.run(source_dir=SOURCE_DIR, out_dir=OUT_DIR)
except Exception:
    readback = OUT_DIR / 'readback.csv'
    if readback.is_file():
        import pandas as pd
        from google.colab import files
        display(pd.read_csv(readback).query('not passed'))
        files.download(str(readback))
    raise
print(json.dumps(paths, ensure_ascii=False, indent=2))

## 3. 전체 성능과 정답 이동
전체142지표를 생략 없이 표시합니다. 차이는 실제 N/V를 사용한 전체 표현에서 비교 상태를 뺀 값입니다. 모든 노출 지표가 클수록 좋은 것은 아닙니다. 가격·구매금액 가중 적중값은 실제 매출이나 미래 CLV가 아닙니다.

정답 이동은 같은 고객·상품과 정답 가중치로 맞추며, 신규 정답 획득뿐 아니라 상실도 보존합니다. 동일고객의 전체 표현 경계상품과 정답의 속성점수 차이도 별도 저장합니다. 미세한 양수나 포화율만으로 새 학습을 결정하지 않습니다.

In [ ]:
import pandas as pd
from IPython.display import display
absolute = pd.read_csv(OUT_DIR / 'absolute.csv').set_index('view')
comparison = pd.read_csv(OUT_DIR / 'comparison.csv')
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(absolute.drop(columns=['seed', 'epoch']).T)
    for reference in diagnostic.VIEWS[1:]:
        print('전체 표현 −', reference)
        display(comparison.query('reference == @reference')[['metric', 'reference_value', 'candidate_value', 'delta', 'relative_change_pct']])
        summary = pd.read_csv(OUT_DIR / ('full_vs_' + reference) / 'summary.csv')
        display(summary[summary.grouping.isin(['all', 'segment'])])
    display(pd.read_csv(OUT_DIR / 'boundary_component_summary.csv'))
result = json.loads((OUT_DIR / 'result.json').read_text())
print(json.dumps(result['mechanism'], ensure_ascii=False, indent=2))
print('유의성·학습 인과효과·자동 다음 실험 판정 없음')

## 4. 결과 ZIP 다운로드
ZIP에는 전체 절대값·비교표·142지표 재현 감사·세 상태의 정답/추천목록·공동순열 배정·두 비교의 정답 이동/CLV세그먼트/상품지지도/가격대·점수 분해·출처 JSON이 들어갑니다. 이후 분석에서는 불리한 결과도 함께 검토합니다. 원본 .pt는 복사하거나 변경하지 않습니다.

In [ ]:
from google.colab import files
files.download(paths['zip'])